### FIFA_WEBAPP

In [1]:
# CELL 1 — Imports
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from scipy.stats import poisson
import pickle
print("Done!")

Done!


In [2]:
# CELL 2 — Load processed data
PATH = 'C:/Users/chand/OneDrive/Desktop/Fifa_Datasets/'
df_combined = pd.read_csv(PATH + 'df_wc_processed.csv')
df_combined['date'] = pd.to_datetime(df_combined['date'], format='mixed')
print(f"Loaded! Shape: {df_combined.shape}")

Loaded! Shape: (7501, 55)


In [3]:
# CELL 3 — Load Poisson model
with open(PATH + 'poisson_model.pkl', 'rb') as f:
    model_data = pickle.load(f)

team_stats    = model_data['team_stats']
avg_goals     = model_data['avg_goals']
avg_home_goals = model_data['avg_home_goals']
avg_away_goals = model_data['avg_away_goals']
home_advantage = model_data['home_advantage']
fifa_rankings  = model_data['fifa_rankings']
wc_groups      = model_data['wc_groups']

print(f"Model loaded!")
print(f"Teams: {len(team_stats)}")
print(f"Avg goals: {avg_goals:.3f}")

Model loaded!
Teams: 48
Avg goals: 1.311


In [4]:
# CELL 4 — Poisson Prediction Function

def predict_match(home_team, away_team, neutral=False):
    if home_team not in team_stats or away_team not in team_stats:
        return None

    home = team_stats[home_team]
    away = team_stats[away_team]

    if neutral:
        home_expected = avg_goals * home['attack_strength'] * away['defence_weakness']
        away_expected = avg_goals * away['attack_strength'] * home['defence_weakness']
    else:
        home_expected = avg_goals * home['attack_strength'] * away['defence_weakness'] * home_advantage
        away_expected = avg_goals * away['attack_strength'] * home['defence_weakness']

    max_goals = 8
    home_win_prob = 0
    draw_prob     = 0
    away_win_prob = 0
    score_matrix  = {}

    for home_goals in range(max_goals + 1):
        for away_goals in range(max_goals + 1):
            prob = (poisson.pmf(home_goals, home_expected) *
                    poisson.pmf(away_goals, away_expected))
            score_matrix[(home_goals, away_goals)] = prob
            if home_goals > away_goals:
                home_win_prob += prob
            elif home_goals == away_goals:
                draw_prob += prob
            else:
                away_win_prob += prob

    most_likely_score = max(score_matrix, key=score_matrix.get)
    most_likely_prob  = score_matrix[most_likely_score]

    return {
        'home_team':              home_team,
        'away_team':              away_team,
        'home_expected_goals':    round(home_expected, 2),
        'away_expected_goals':    round(away_expected, 2),
        'home_win_prob':          round(home_win_prob * 100, 1),
        'draw_prob':              round(draw_prob * 100, 1),
        'away_win_prob':          round(away_win_prob * 100, 1),
        'most_likely_score':      f"{most_likely_score[0]}-{most_likely_score[1]}",
        'most_likely_score_prob': round(most_likely_prob * 100, 1),
    }

# Quick test
result = predict_match('Argentina', 'France', neutral=True)
print(f"Argentina vs France")
print(f"Argentina Win : {result['home_win_prob']}%")
print(f"Draw          : {result['draw_prob']}%")
print(f"France Win    : {result['away_win_prob']}%")
print(f"Most likely   : {result['most_likely_score']}")

Argentina vs France
Argentina Win : 35.2%
Draw          : 29.0%
France Win    : 35.8%
Most likely   : 1-1


In [5]:
# CELL 2 — Load processed data
PATH = 'C:/Users/chand/OneDrive/Desktop/Fifa_Datasets/'
df_wc = pd.read_csv(PATH + 'df_wc_processed.csv')
df_wc['date'] = pd.to_datetime(df_wc['date'], format='mixed')
print(f"Loaded! Shape: {df_wc.shape}")

Loaded! Shape: (7501, 55)


In [6]:
# CELL 3 — Load Poisson model
with open(PATH + 'poisson_model.pkl', 'rb') as f:
    model_data = pickle.load(f)

team_stats     = model_data['team_stats']
avg_goals      = model_data['avg_goals']
avg_home_goals = model_data['avg_home_goals']
avg_away_goals = model_data['avg_away_goals']
home_advantage = model_data['home_advantage']
fifa_rankings  = model_data['fifa_rankings']
wc_groups      = model_data['wc_groups']

print(f"Model loaded! Teams: {len(team_stats)}")

Model loaded! Teams: 48


---

#### 7501 rows = 7501 historical matches between 48 WC teams
#### 55 columns = all our engineered features

In [10]:
df_wc.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,year,...,away_confederation,home_conf_strength,away_conf_strength,conf_strength_diff,same_confederation,poisson_home_win,poisson_draw,poisson_away_win,poisson_home_exp,poisson_away_exp
0,1872-11-30,Scotland,England,0,0,Friendly,Glasgow,Scotland,0,1872,...,UEFA,5,5,0,1,0.189,0.198,0.613,1.16,2.21
1,1873-03-08,England,Scotland,4,2,Friendly,London,England,0,1873,...,UEFA,5,5,0,1,0.771,0.136,0.090,2.84,0.90
2,1874-03-07,Scotland,England,2,1,Friendly,Glasgow,Scotland,0,1874,...,UEFA,5,5,0,1,0.189,0.198,0.613,1.16,2.21
3,1875-03-06,England,Scotland,2,2,Friendly,London,England,0,1875,...,UEFA,5,5,0,1,0.771,0.136,0.090,2.84,0.90
4,1876-03-04,Scotland,England,3,0,Friendly,Glasgow,Scotland,0,1876,...,UEFA,5,5,0,1,0.189,0.198,0.613,1.16,2.21


---

#### The web app only needs:
#### poisson_model.pkl → has team_stats, avg_goals, home_advantage
#### predict_match() function → calculates predictions
#### That's it. User enters two teams → function runs → shows probabilities.